# Reproducibility notebook for the corrected beech phenology article

This notebook is the main entry point for the public reproducibility repository that accompanies the corrected article on European beech spring phenology in Slovenia.

## How this notebook maps to the manuscript

- Result 1 - Section 3.1 - Agreement between satellite-derived SOS and in situ observations
- Result 2 - Section 3.2 - Altitudinal sensitivity of spring onset
- Result 3 - Section 3.3 - Spatial patterns across ecological regions
- Result 4 - Section 3.4 - Interannual variability in national mean SOS
- Result 5 - Section 3.5 - Association between SOS variability and spring temperature

The section titles below intentionally repeat the manuscript structure so that readers can see exactly which code block supports which part of the article.

## Data availability and repository scope

This notebook is aligned with the article's Data availability statement.

- Annual SOS raster maps for European beech in Slovenia, derived from Sentinel-2 IRECI time series for 2018-2021, are publicly available at `https://doi.org/10.5281/zenodo.20358835`.
- The accompanying Python code and reproducibility sample documenting Sentinel-2 time-series preprocessing, vegetation index calculation, temporal smoothing, SOS extraction, and generation of yearly SOS tables are available at `https://github.com/Anapot/phenology-timeseries`.
- The complete Sentinel-2 national time-series archive is not redistributed.

This repository therefore starts from the public SOS raster products and from lightweight publishable analysis inputs, and it focuses specifically on reproducing the corrected article's analyses.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
from rasterio.features import rasterize
from rasterio.warp import Resampling, reproject
from scipy.stats import linregress

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

YEARS = [2018, 2019, 2020, 2021]
SOS_PATTERN = "sos_IRECI_{year}_uint16_compressed.tif"
SOS_MIN = 50
SOS_MAX = 180
REGION_COLUMN = "Region"


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the reproducibility repository root.")


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def path_status(path: Path) -> str:
    return "OK" if path.exists() else "MISSING"


def load_dem_on_grid(dem_path: Path, template_src) -> np.ndarray:
    dem_resampled = np.empty(template_src.shape, dtype=np.float32)
    with rasterio.open(dem_path) as dem_src:
        reproject(
            source=rasterio.band(dem_src, 1),
            destination=dem_resampled,
            src_transform=dem_src.transform,
            src_crs=dem_src.crs,
            dst_transform=template_src.transform,
            dst_crs=template_src.crs,
            resampling=Resampling.bilinear,
        )
    return dem_resampled


def build_region_lookup(regions: gpd.GeoDataFrame):
    regions = regions.copy().reset_index(drop=True)
    regions["region_id"] = np.arange(1, len(regions) + 1)
    return regions, dict(zip(regions["region_id"], regions[REGION_COLUMN]))


def rasterize_regions(regions: gpd.GeoDataFrame, src) -> np.ndarray:
    regions_in_crs = regions.to_crs(src.crs)
    shapes = [
        (geom, region_id)
        for geom, region_id in zip(regions_in_crs.geometry, regions_in_crs["region_id"])
        if geom is not None
    ]
    return rasterize(
        shapes=shapes,
        out_shape=src.shape,
        transform=src.transform,
        fill=0,
        dtype="int32",
    )


repo_root = find_repo_root(Path.cwd())
data_dir = repo_root / "data"
external_dir = data_dir / "external"
processed_dir = data_dir / "processed"
reference_dir = data_dir / "reference"
outputs_figures_dir = ensure_dir(repo_root / "outputs" / "figures")
outputs_tables_dir = ensure_dir(repo_root / "outputs" / "tables")

sos_layers_dir = external_dir / "zenodo_sos_layers"
dem_dir = external_dir / "public_dem"
dem_path = dem_dir / "copernicus_dem_slovenia.tif"
agreement_points_path = processed_dir / "agreement_sos_publishable_points_dedup.csv"
agreement_metrics_archive_path = processed_dir / "agreement_sos_publishable_metrics_dedup.csv"
temperature_table_path = processed_dir / "regional_temperature_sos_table.csv"
regions_path = reference_dir / "Ekoloske_regije.gpkg"

print(f"Repository root: {repo_root}")
print(f"SOS raster folder: {sos_layers_dir} [{path_status(sos_layers_dir)}]")
print(f"Public DEM: {dem_path} [{path_status(dem_path)}]")
print(f"Agreement points table: {agreement_points_path} [{path_status(agreement_points_path)}]")
print(f"Agreement metrics archive: {agreement_metrics_archive_path} [{path_status(agreement_metrics_archive_path)}]")
print(f"Temperature table: {temperature_table_path} [{path_status(temperature_table_path)}]")
print(f"Ecological regions layer: {regions_path} [{path_status(regions_path)}]")

## Input configuration for this repository

The notebook is designed so that every result section states its own inputs explicitly.

- Results 2 and 3 require external files downloaded by the reader: Zenodo SOS rasters and a public DEM.
- Result 1 uses the bundled public agreement table in `data/processed/`.
- Result 5 uses the bundled regional temperature-SOS table in `data/processed/`.
- Result 4 is computed from the annual regional SOS table generated in Result 3.

## Result 1 - Section 3.1 - Agreement between satellite-derived SOS and in situ observations

This section reproduces the public agreement figure and summary metrics from the publishable, anonymized agreement table bundled with this repository.

In [ ]:
if not agreement_points_path.exists():
    raise FileNotFoundError(f"Missing agreement table: {agreement_points_path}")

agreement_df = pd.read_csv(agreement_points_path)
indices = ["IRECI", "NDVI", "EVI"]

def agreement_metrics(df: pd.DataFrame) -> dict:
    slope, intercept, r_value, p_value, std_err = linregress(df["DOY_insitu"], df["DOY_model"])
    residuals = df["DOY_model"] - df["DOY_insitu"]
    return {
        "n": int(len(df)),
        "rmse": float(np.sqrt(np.mean(np.square(residuals)))),
        "bias": float(np.mean(residuals)),
        "r": float(r_value),
        "r2": float(r_value ** 2),
        "slope": float(slope),
        "intercept": float(intercept),
        "p_value": float(p_value),
        "standard_error": float(std_err),
    }

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.6), sharex=True, sharey=True)
summary_rows = []
all_values = agreement_df[["DOY_insitu", "DOY_model"]].to_numpy().ravel()
axis_min = np.floor(all_values.min() / 5) * 5
axis_max = np.ceil(all_values.max() / 5) * 5

for axis, index_name in zip(axes, indices):
    df_index = agreement_df.loc[agreement_df["index"] == index_name].copy()
    metrics = agreement_metrics(df_index)
    summary_rows.append({"index": index_name, **metrics})

    axis.scatter(df_index["DOY_insitu"], df_index["DOY_model"], s=16, alpha=0.75)
    axis.plot([axis_min, axis_max], [axis_min, axis_max], linestyle="--", linewidth=1.0)
    line_x = np.linspace(axis_min, axis_max, 100)
    axis.plot(line_x, metrics["intercept"] + metrics["slope"] * line_x, linewidth=1.1)
    axis.set_title(index_name)
    axis.set_xlim(axis_min, axis_max)
    axis.set_ylim(axis_min, axis_max)
    axis.set_xlabel("Observed SOS (DOY)")
    axis.text(
        0.03,
        0.97,
        f"R2 = {metrics['r2']:.2f}\nRMSE = {metrics['rmse']:.1f}\nBias = {metrics['bias']:.1f}\nn = {metrics['n']}",
        transform=axis.transAxes,
        va="top",
        ha="left",
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
    )

axes[0].set_ylabel("Satellite-derived SOS (DOY)")
plt.tight_layout()

agreement_figure_path = outputs_figures_dir / "Figure_3_agreement_satellite_vs_insitu.pdf"
agreement_metrics_path = outputs_tables_dir / "Table_Result1_agreement_metrics.csv"
plt.savefig(agreement_figure_path, format="pdf", bbox_inches="tight")
plt.show()

agreement_metrics_df = pd.DataFrame(summary_rows)
agreement_metrics_df.to_csv(agreement_metrics_path, index=False)
agreement_metrics_df

## Result 2 - Section 3.2 - Altitudinal sensitivity of spring onset

This section uses the public Zenodo SOS rasters plus a public DEM to quantify the elevation dependence of SOS and to regenerate the main elevation-response figures.

If a DEM different from the original internal DEM is used, small numerical differences are expected. The analytical logic stays the same.

In [ ]:
if not sos_layers_dir.exists():
    raise FileNotFoundError(f"Missing SOS raster folder: {sos_layers_dir}")
if not dem_path.exists():
    raise FileNotFoundError(f"Missing DEM file: {dem_path}")

altitude_summary_rows = []
bin_frames = []
fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.8), sharex=True, sharey=True)
axes = axes.flatten()

for axis, year in zip(axes, YEARS):
    sos_path = sos_layers_dir / SOS_PATTERN.format(year=year)
    if not sos_path.exists():
        raise FileNotFoundError(f"Missing SOS raster: {sos_path}")

    with rasterio.open(sos_path) as src:
        sos = src.read(1).astype(np.float32)
        dem = load_dem_on_grid(dem_path, src)
        valid_mask = np.isfinite(sos) & np.isfinite(dem) & (sos >= SOS_MIN) & (sos <= SOS_MAX)
        if src.nodata is not None:
            valid_mask &= sos != src.nodata

        sos_valid = sos[valid_mask].astype(float)
        elev_valid = dem[valid_mask].astype(float)
        elev_100m = elev_valid / 100.0
        slope, intercept, r_value, p_value, std_err = linregress(elev_100m, sos_valid)

        altitude_summary_rows.append({
            "year": year,
            "n_pixels": int(len(sos_valid)),
            "slope_days_per_100m": float(slope),
            "intercept": float(intercept),
            "r2": float(r_value ** 2),
            "p_value": float(p_value),
            "sos_min": float(sos_valid.min()),
            "sos_max": float(sos_valid.max()),
        })

        alt_df = pd.DataFrame({"elevation": elev_valid, "sos": sos_valid})
        alt_df["elev_bin"] = (alt_df["elevation"] // 200) * 200
        bins_df = (
            alt_df.groupby("elev_bin", observed=True)
            .agg(mean_sos=("sos", "mean"), n_pixels=("sos", "count"), mean_elevation=("elevation", "mean"))
            .reset_index()
        )
        bins_df["year"] = year
        bin_frames.append(bins_df)

        axis.hexbin(elev_100m, sos_valid, gridsize=60, mincnt=1)
        x_line = np.linspace(elev_100m.min(), elev_100m.max(), 100)
        axis.plot(x_line, intercept + slope * x_line, linewidth=1.1)
        axis.set_title(str(year))
        axis.set_xlabel("Elevation (100 m)")
        axis.set_ylabel("SOS (DOY)")
        axis.text(
            0.03,
            0.97,
            f"beta = {slope:.2f} d / 100 m\nR2 = {r_value ** 2:.2f}",
            transform=axis.transAxes,
            va="top",
            ha="left",
            bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
        )

plt.tight_layout()
altitude_panel_path = outputs_figures_dir / "Figure_4_altitudinal_sensitivity_panels.pdf"
plt.savefig(altitude_panel_path, format="pdf", bbox_inches="tight")
plt.show()

altitude_summary_df = pd.DataFrame(altitude_summary_rows)
altitude_summary_df.to_csv(outputs_tables_dir / "Table_Result2_altitudinal_sensitivity.csv", index=False)

bin_summary_df = pd.concat(bin_frames, ignore_index=True)
fig, ax = plt.subplots(figsize=(5.3, 3.8))
for year, year_df in bin_summary_df.groupby("year"):
    ax.plot(year_df["mean_elevation"], year_df["mean_sos"], marker="o", linewidth=1.0, markersize=3, label=str(year))
ax.set_xlabel("Mean elevation of 200 m bin (m)")
ax.set_ylabel("Mean SOS (DOY)")
ax.legend(title="Year")
plt.tight_layout()
plt.savefig(outputs_figures_dir / "Figure_4_altitudinal_bin_curves.pdf", format="pdf", bbox_inches="tight")
plt.show()

altitude_summary_df

## Result 3 - Section 3.3 - Spatial patterns across ecological regions

This section regenerates the regional SOS summary tables and the associated inferential statistics. It also fits the weighted elevation-plus-region regression using aggregated elevation bins so that the public workflow stays memory-safe.

In [ ]:
if not regions_path.exists():
    raise FileNotFoundError(f"Missing ecological-region layer: {regions_path}")

regions = gpd.read_file(regions_path)
if REGION_COLUMN not in regions.columns:
    raise ValueError(f"Column '{REGION_COLUMN}' is missing from {regions_path}")
regions = regions.loc[regions.geometry.notnull()].copy()
regions, id_to_region = build_region_lookup(regions)

annual_rows = []
agg_frames = []

for year in YEARS:
    sos_path = sos_layers_dir / SOS_PATTERN.format(year=year)
    with rasterio.open(sos_path) as src:
        sos = src.read(1).astype(np.float32)
        dem = load_dem_on_grid(dem_path, src)
        region_raster = rasterize_regions(regions, src)

        valid_mask = (region_raster > 0) & np.isfinite(sos) & np.isfinite(dem) & (sos >= SOS_MIN) & (sos <= SOS_MAX)
        if src.nodata is not None:
            valid_mask &= sos != src.nodata

        sos_valid = sos[valid_mask].astype(float)
        dem_valid = dem[valid_mask].astype(float)
        region_valid = region_raster[valid_mask]

        year_df = pd.DataFrame({
            "region_id": region_valid,
            "sos": sos_valid,
            "elevation": dem_valid,
        })

        region_stats = (
            year_df.groupby("region_id", observed=True)
            .agg(
                n_pixels=("sos", "count"),
                mean_sos=("sos", "mean"),
                min_pixel_sos=("sos", "min"),
                max_pixel_sos=("sos", "max"),
                sd_pixel_sos=("sos", "std"),
            )
            .reset_index()
        )
        region_stats["year"] = year
        region_stats["Region"] = region_stats["region_id"].map(id_to_region)
        annual_rows.append(region_stats)

        year_df["year"] = str(year)
        year_df["Region"] = pd.Series(region_valid).map(id_to_region).astype(str).values
        year_df["elev_bin"] = (year_df["elevation"] // 50) * 50
        agg_df = (
            year_df.groupby(["year", "Region", "elev_bin"], observed=True)
            .agg(
                n_pixels=("sos", "count"),
                mean_sos=("sos", "mean"),
                mean_elevation=("elevation", "mean"),
            )
            .reset_index()
        )
        agg_frames.append(agg_df)

annual_df = pd.concat(annual_rows, ignore_index=True)
annual_df = annual_df[["Region", "region_id", "year", "n_pixels", "mean_sos", "min_pixel_sos", "max_pixel_sos", "sd_pixel_sos"]]
annual_table_path = outputs_tables_dir / "Table_SOS_ecological_regions_annual.csv"
annual_df.to_csv(annual_table_path, index=False)

summary_df = (
    annual_df.groupby("Region", observed=True)
    .agg(
        N_pixels=("n_pixels", "sum"),
        Mean_SOS=("mean_sos", "mean"),
        Minimum=("mean_sos", "min"),
        Maximum=("mean_sos", "max"),
        SD_years=("mean_sos", "std"),
    )
    .reset_index()
)
summary_df["Range"] = summary_df["Maximum"] - summary_df["Minimum"]
summary_df["CV_percent"] = (summary_df["SD_years"] / summary_df["Mean_SOS"]) * 100.0
summary_df = summary_df.round({
    "Mean_SOS": 1,
    "Minimum": 1,
    "Maximum": 1,
    "SD_years": 1,
    "Range": 1,
    "CV_percent": 1,
})
summary_table_path = outputs_tables_dir / "Table_SOS_ecological_regions.csv"
summary_df.to_csv(summary_table_path, index=False)

anova_input = annual_df.copy()
anova_input["year"] = anova_input["year"].astype(str)
anova_model = smf.ols("mean_sos ~ C(Region) + C(year)", data=anova_input).fit()
anova_table = sm.stats.anova_lm(anova_model, typ=2)
anova_table.to_csv(outputs_tables_dir / "Table_Result3_anova.csv")

agg_df = pd.concat(agg_frames, ignore_index=True)
reference_region = "Alpine"
region_levels = list(agg_df["Region"].drop_duplicates())
if reference_region not in region_levels:
        reference_region = region_levels[0]
agg_df["Region"] = pd.Categorical(
    agg_df["Region"],
    categories=[reference_region] + [value for value in region_levels if value != reference_region],
)
pixel_model = smf.wls(
    "mean_sos ~ mean_elevation + C(Region) + C(year)",
    data=agg_df,
    weights=agg_df["n_pixels"],
).fit()

coefficients_df = pixel_model.params.rename("coefficient").to_frame()
coefficients_df["lower_95"] = pixel_model.conf_int()[0]
coefficients_df["upper_95"] = pixel_model.conf_int()[1]
coefficients_df.to_csv(outputs_tables_dir / "Table_Result3_weighted_regression_coefficients.csv")

print("Two-way ANOVA on regional mean SOS values")
display(anova_table)
print("Weighted aggregated regression summary")
print(pixel_model.summary())

summary_df

## Result 4 - Section 3.4 - Interannual variability in national mean SOS

This section derives the national annual anomaly table directly from the annual ecological-region table generated in Result 3.

In [ ]:
if "annual_df" not in globals():
    annual_df = pd.read_csv(outputs_tables_dir / "Table_SOS_ecological_regions_annual.csv")

national_df = (
    annual_df.groupby("year", observed=True)
    .agg(Mean_SOS_DOY=("mean_sos", "mean"))
    .reset_index()
)
reference_mean = national_df["Mean_SOS_DOY"].mean()
national_df["Reference_mean_DOY"] = reference_mean
national_df["Anomaly_DOY"] = national_df["Mean_SOS_DOY"] - reference_mean
national_df = national_df.round({"Mean_SOS_DOY": 2, "Reference_mean_DOY": 2, "Anomaly_DOY": 2})
national_table_path = outputs_tables_dir / "Table_National_mean_SOS_anomalies.csv"
national_df.to_csv(national_table_path, index=False)
national_df

## Result 5 - Section 3.5 - Association between SOS variability and spring temperature

This section reproduces the regional spring-temperature versus mean-SOS analysis from the bundled lightweight regional year table.

In [ ]:
if not temperature_table_path.exists():
    raise FileNotFoundError(f"Missing temperature table: {temperature_table_path}")

temperature_df = pd.read_csv(temperature_table_path)
slope, intercept, r_value, p_value, std_err = linregress(temperature_df["April"], temperature_df["Mean_SOS"])
r2 = r_value ** 2

fig, ax = plt.subplots(figsize=(3.6, 3.2))
ax.scatter(temperature_df["April"], temperature_df["Mean_SOS"], s=25, alpha=0.8)
x_line = np.linspace(temperature_df["April"].min() - 0.5, temperature_df["April"].max() + 0.5, 100)
ax.plot(x_line, intercept + slope * x_line, linewidth=1.1)
ax.set_xlabel("Mean April temperature (deg C)")
ax.set_ylabel("Mean SOS (DOY)")
ax.text(
    0.03,
    0.03,
    f"R2 = {r2:.2f}\nbeta = {slope:.2f} d / deg C\np = {p_value:.3g}\nn = {len(temperature_df)}",
    transform=ax.transAxes,
    va="bottom",
    ha="left",
    bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
)
plt.tight_layout()
temperature_figure_path = outputs_figures_dir / "Figure_5_temperature_vs_sos.pdf"
plt.savefig(temperature_figure_path, format="pdf", bbox_inches="tight")
plt.show()

temperature_summary_df = pd.DataFrame([
    {
        "slope_days_per_degC": slope,
        "intercept": intercept,
        "r": r_value,
        "r2": r2,
        "p_value": p_value,
        "standard_error": std_err,
        "n": len(temperature_df),
    }
])
temperature_summary_df.to_csv(outputs_tables_dir / "Table_Result5_temperature_regression.csv", index=False)
temperature_summary_df

## Notes on manuscript consistency

- Results 2 and 3 should be interpreted as direct reproductions of the published analysis logic from the public SOS rasters.
- Small numerical differences may appear if the reader uses a public DEM different from the internal DEM used during manuscript preparation.
- Result 1 uses the public anonymized agreement table bundled here because raw in situ observations are not redistributed.
- Result 5 uses a bundled lightweight regional year table because this repository is focused on analysis reproducibility, not on rebuilding the full upstream climate preprocessing chain.